## Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys

# Define the path to the root of your cloned repository in Google Drive
# IMPORTANT: Make sure this path is correct based on where you cloned the repo
repo_root = '/home/coder/project/quantization_techniques/quantization_techniques'

# Add the repository root to the Python path
if repo_root not in sys.path:
    sys.path.append(repo_root)
    print(f"Added {repo_root} to sys.path")
else:
    print(f"{repo_root} is already in sys.path")

Added /home/coder/project/quantization_techniques/quantization_techniques to sys.path


In [3]:
import torch
from tqdm import tqdm
from copy import deepcopy

from ADC.conv_experiments.conv_experiment_setup import get_config, get_exp_config, setup_dataloaders
from ADC.conv_experiments.conv_experiment import run_single_experiment
from ADC.models import resnet18_cifar, resnet18_cifar_adc
from ADC.train_utils import calibrate_model, train_model
from ADC.logger import LayerwiseStatsLogger

from ADC.quantized_layers import TiledConv2dADC, Conv2dADC
from torch import nn

def load_weights_with_tiling_support(custom_model, pretrained_model, verbose=False):
    pretrained_modules = dict(pretrained_model.named_modules())
    custom_modules = dict(custom_model.named_modules())

    for name, custom_module in custom_modules.items():
        if name in pretrained_modules:
            pretrained_module = pretrained_modules[name]

            # Special case for TiledConv2dADC
            if isinstance(custom_module, TiledConv2dADC) and isinstance(pretrained_module, nn.Conv2d):
                if verbose:
                    print(f"[TILED] Loading weights into: {name}")
                custom_module.load_weights(pretrained_module)

            # Default case: try to load parameters directly
            else:
                try:
                    custom_module.load_state_dict(pretrained_module.state_dict(), strict=False)
                    if verbose:
                        print(f"[DEFAULT] Loaded state_dict for: {name}")
                except RuntimeError as e:
                    if verbose:
                        print(f"[WARNING] Skipped {name} due to mismatch: {e}")

def evaluate_model(model, test_loader):
    criterion = torch.nn.CrossEntropyLoss()
    model.eval()  # Set the model to evaluation mode
    correct = 0
    total = 0
    total_loss = 0.
    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc=f"Calculating accuracy"):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100. * correct / total, total_loss / total

In [4]:
train_loader, test_loader = setup_dataloaders(256, 256)

In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [6]:
baseline_weights = "/home/coder/project/quantization_techniques/model_R18_weights_20250616_220000.pth"
model_baseline = resnet18_cifar()
model_baseline.load_state_dict(torch.load(baseline_weights))
model_baseline = model_baseline.to(device)

In [12]:
logger = LayerwiseStatsLogger()
model_adc = resnet18_cifar_adc(num_classes=10, bx=8, bw=8, ba=8, k=4, ashift=False, logger=None).to(device)
load_weights_with_tiling_support(model_adc, model_baseline)
#model_adc.load_state_dict(torch.load(baseline_weights), strict=False)
#model_adc.disable_adc()
model_adc.enable_adc()
#model_adc.conv1.enable_adc()

enable adc


In [14]:
calibrate_model(model_adc, train_loader, device)
evaluate_model(model_adc, test_loader)

Calibrating quantizers...
Calibration done. Quantizer observers are now disabled.


Calculating accuracy: 100%|██████████| 40/40 [00:03<00:00, 13.02it/s]


(31.51, 2.4751140674591063)

In [20]:
logger = LayerwiseStatsLogger()
model_adc2 = resnet18_cifar_adc(num_classes=10, bx=4, bw=4, ba=8, k=4, ashift=False, logger=None, conv_type=TiledConv2dADC).to(device)
load_weights_with_tiling_support(model_adc2, model_baseline)
#model_adc.load_state_dict(torch.load(baseline_weights), strict=False)
#model_adc.disable_adc()
model_adc2.enable_adc()
#model_adc.conv1.enable_adc()

enable adc


In [23]:
calibrate_model(model_adc2, train_loader, device)
evaluate_model(model_adc2, test_loader)

Calibrating quantizers...
Calibration done. Quantizer observers are now disabled.


Calculating accuracy: 100%|██████████| 40/40 [00:04<00:00,  8.49it/s]


(70.75, 1.0226531281471252)

In [ ]:
torch.sum([torch.normal(0, 1, size=(2, 3)), torch.normal(0, 1, size=(2, 3))])

tensor([[ 0.4199,  2.3655, -0.7076],
        [-1.0611, -0.7371, -0.2599]])

In [ ]:
model_adc2 = resnet18_cifar_adc(num_classes=10, bx=8, bw=8, ba=8, k=32, ashift=False, logger=None, conv_type=TiledConv2dADC).to(device)

In [16]:
load_weights_with_tiling_support(model_adc, model_baseline)

[DEFAULT] Loaded state_dict for: bn1
[DEFAULT] Loaded state_dict for: relu
[DEFAULT] Loaded state_dict for: layer1
[DEFAULT] Loaded state_dict for: layer1.0.bn1
[DEFAULT] Loaded state_dict for: layer1.0.bn2
[DEFAULT] Loaded state_dict for: layer1.0.relu
[DEFAULT] Loaded state_dict for: layer1.1.bn1
[DEFAULT] Loaded state_dict for: layer1.1.bn2
[DEFAULT] Loaded state_dict for: layer1.1.relu
[DEFAULT] Loaded state_dict for: layer2
[DEFAULT] Loaded state_dict for: layer2.0.bn1
[DEFAULT] Loaded state_dict for: layer2.0.bn2
[DEFAULT] Loaded state_dict for: layer2.0.relu
[DEFAULT] Loaded state_dict for: layer2.0.downsample
[DEFAULT] Loaded state_dict for: layer2.0.downsample.1
[DEFAULT] Loaded state_dict for: layer2.1.bn1
[DEFAULT] Loaded state_dict for: layer2.1.bn2
[DEFAULT] Loaded state_dict for: layer2.1.relu
[DEFAULT] Loaded state_dict for: layer3
[DEFAULT] Loaded state_dict for: layer3.0.bn1
[DEFAULT] Loaded state_dict for: layer3.0.bn2
[DEFAULT] Loaded state_dict for: layer3.0.relu
[D

In [ ]:
model_adc2 = resnet18_cifar_adc(num_classes=10, bx=8, bw=8, ba=8, k=32, ashift=False, logger=logger).to(device)

In [ ]:
# import gdown
# file_id = "1VIJn_HH6tNTbBhYopXS0uH5RTVSCH6To"
# url = f'https://drive.google.com/uc?id={file_id}'
# output = 'model_R18_weights_20250616_220000.pth'

# gdown.download(url, output, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1VIJn_HH6tNTbBhYopXS0uH5RTVSCH6To
From (redirected): https://drive.google.com/uc?id=1VIJn_HH6tNTbBhYopXS0uH5RTVSCH6To&confirm=t&uuid=23b26909-e2ef-42f6-9dd6-e08bc9e3de6d
To: /home/coder/project/quantization_techniques/model_R18_weights_20250616_220000.pth
100%|██████████| 44.8M/44.8M [00:01<00:00, 34.8MB/s]


'model_R18_weights_20250616_220000.pth'

In [38]:
results = [torch.normal(0, 1, size=(3, 4, 5)) for i in range(6)]

In [39]:
torch.cat(results, axis=0).shape

torch.Size([18, 4, 5])

In [45]:
[t.shape for t in torch.split(torch.normal(0, 1, size=(2,18, 4, 5)), 5, dim=1)]

[torch.Size([2, 5, 4, 5]),
 torch.Size([2, 5, 4, 5]),
 torch.Size([2, 5, 4, 5]),
 torch.Size([2, 3, 4, 5])]